In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l7.5 Preference optimization

SFT teaches *a* good answer; preference methods teach *which of two* answers humans
prefer. **DPO** does this directly on `(chosen, rejected)` pairs against a frozen
reference model, no separate reward model or RL loop.

In [ ]:
from pocketlm import dpo_loss

ref_chosen, ref_rejected = torch.tensor(-2.0), torch.tensor(-2.0)
good = dpo_loss(torch.tensor(-1.0), torch.tensor(-3.0), ref_chosen, ref_rejected)
bad = dpo_loss(torch.tensor(-3.0), torch.tensor(-1.0), ref_chosen, ref_rejected)
print("loss when policy prefers the chosen reply: ", round(good.item(), 3))
print("loss when policy prefers the rejected reply:", round(bad.item(), 3))

DPO's loss is low exactly when the policy raises the chosen-over-rejected margin
beyond the reference. Optimizing it nudges the model toward human-preferred
responses, the alignment step after SFT.

In [ ]:
assert good < bad